# Notebook 5 - Machine Learning
Train baseline models for next-day price direction prediction.

In [ ]:
# Install if required
# !pip install scikit-learn xgboost lightgbm

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE=True
except:
    XGB_AVAILABLE=False

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE=True
except:
    LGBM_AVAILABLE=False


In [ ]:
df=pd.read_parquet("processed/feature_engineered_data.parquet")

# Target: 1 if next day's close is higher than today's close
if "Price_Direction" in df.columns:
    target="Price_Direction"
else:
    df["Next_Close"]=df.groupby("Symbol")["Close"].shift(-1)
    df["Price_Direction"]=(df["Next_Close"]>df["Close"]).astype(int)
    target="Price_Direction"

feature_cols=[
    c for c in [
        "Open","High","Low","Close","Volume",
        "Daily_Return","Log_Return",
        "SMA_5","SMA_20","SMA_50",
        "EMA_12","EMA_26",
        "MACD","RSI",
        "Rolling_Volatility",
        "Volume_Change",
        "Lag_Close_1","Lag_Close_2","Lag_Close_3"
    ] if c in df.columns
]

X=df[feature_cols]
y=df[target]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y)


In [ ]:
models={
"Logistic Regression":Pipeline([
("imputer",SimpleImputer(strategy="median")),
("model",LogisticRegression(max_iter=1000))
]),
"Random Forest":Pipeline([
("imputer",SimpleImputer(strategy="median")),
("model",RandomForestClassifier(n_estimators=200,random_state=42))
])
}

if XGB_AVAILABLE:
    models["XGBoost"]=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("model",XGBClassifier(eval_metric="logloss"))
    ])

if LGBM_AVAILABLE:
    models["LightGBM"]=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("model",LGBMClassifier())
    ])

results=[]

for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)

    results.append({
        "Model":name,
        "Accuracy":accuracy_score(y_test,pred),
        "Precision":precision_score(y_test,pred,zero_division=0),
        "Recall":recall_score(y_test,pred,zero_division=0),
        "F1":f1_score(y_test,pred,zero_division=0)
    })

    print("="*60)
    print(name)
    print(classification_report(y_test,pred))
    print(confusion_matrix(y_test,pred))

results_df=pd.DataFrame(results).sort_values("Accuracy",ascending=False)
results_df


In [ ]:
results_df.to_csv("processed/ml_model_comparison.csv",index=False)
print("Saved processed/ml_model_comparison.csv")


## Next Steps

- Hyperparameter tuning
- Time-series cross validation
- Feature importance
- SHAP explainability
- LSTM model
- Trade settlement risk model
